Concepts
1. Basic Cuda Code
2. threadIdx
3. Compute global index
4. Error handling
5. Global memory v/s Shared memory

In [ ]:
%%writefile add.cu
#include <stdio.h>

__global__
void add(int a , int b , int *result){
  *result = a + b ;
}

int main(){
  int result ;
  int *d_result ;

  cudaMalloc(&d_result , sizeof(int)); // allocate on GPU
  add<<<1,1>>>(2,3,d_result);         // run on GPU
  cudaMemcpy(&result , d_result , sizeof(int) , cudaMemcpyDeviceToHost);
  printf("2 + 3 = %d\n",result);
  cudaFree(d_result);

  return 0 ;
}


In [ ]:
!nvcc add.cu -o add && ./add

In [ ]:
%%writefile thread.cu

#include <stdio.h>
#include <cuda_runtime.h>

__global__ void whoAmI(){
  printf("I am thread %d\n",threadIdx.x);
}

int main(){
  whoAmI<<<1 , 6>>>();

  cudaError_t err = cudaGetLastError();
  if (err != cudaSuccess){
    printf("Kernel launch error: %s\n",cudaGetErrorString(err));
    return -1 ;
  }

  cudaDeviceSynchronize(); // waits for gpu to finish
  return 0 ;
}

In [ ]:
!nvcc thread.cu -o thread && ./thread

In [ ]:
%%writefile sharedMem.cu

#include <stdio.h>
#include <cuda_runtime.h>

__global__ void example (int *arr , int n){
  __shared__ int title[4];

  int i = blockIdx.x * blockDim.x + threadIdx.x;

  title[threadIdx.x] = arr[i];
  __syncthreads();

  arr[i] = title[threadIdx.x] * 2;
}

int main(){
  int arr[10] = {1,2,3,4,5,6,7,8,9,10};
  int *d_arr ; 
  int n = 10 ;

  cudaMalloc(&d_arr , n*sizeof(int));
  cudaMemcpy(d_arr , arr , n * sizeof(int) , cudaMemcpyHostToDevice);
  example<<<2 ,5>>>(d_arr , n);
  cudaMemcpy(arr , d_arr , n * sizeof(int) , cudaMemcpyDeviceToHost);
  
   for (int i = 0 ; i < n ; i++){
    printf("%d\n",arr[i]);
  }
  cudaFree(d_arr);
  return 0 ;
}

In [ ]:
!nvcc sharedMem.cu -o sharedMem && ./sharedMem